# Preprocesamiento del dataset COMPAS

<!-- Este notebook:
1. Carga el dataset COMPAS.
2. Aplica el preprocesamiento visto hasta ahora.
3. Genera archivos `.csv` listos para usar en entrenamiento.

Incluye:
- limpieza básica,
- filtrado de filas inválidas,
- selección de los subconjuntos de 2, 7 y 8 características,
- codificación de variables categóricas,
- división train/test,
- guardado de archivos finales. -->


In [2]:
# Dependencias

from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder


## 1. Configuración

In [3]:
RAW_DIR = Path("data") / "raw"
PROCESSED_DIR = Path("data") / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Nombre esperado del archivo original
RAW_FILE = RAW_DIR / "compas-scores-two-years.csv"

# Salidas
PROCESSED_FILE = PROCESSED_DIR / "compas_preprocessed.csv"
TRAIN_FILE = PROCESSED_DIR / "compas_preprocessed_train.csv"
TEST_FILE = PROCESSED_DIR / "compas_preprocessed_test.csv"
META_FILE = PROCESSED_DIR / "compas_preprocessing_metadata.json"


## 2. Carga del dataset

In [4]:
def locate_csv(raw_dir: Path) -> Path:
    candidates = sorted(raw_dir.rglob("*.csv"))
    if not candidates:
        raise FileNotFoundError(
            f"No encontré CSV en {raw_dir}. Coloca ahí 'compas-scores-two-years.csv'."
        )
    for p in candidates:
        if p.name.lower() == "compas-scores-two-years.csv":
            return p
    return candidates[0]

if RAW_FILE.exists():
    csv_path = RAW_FILE
else:
    csv_path = locate_csv(RAW_DIR)

df = pd.read_csv(csv_path)
print("Archivo cargado:", csv_path)
print("Forma original:", df.shape)
display(df.head())


Archivo cargado: data\raw\compas-scores-two-years.csv
Forma original: (7214, 53)


,id,name,first,last,compas_screening_date,sex,dob,age,age_cat,race,...,v_decile_score,v_score_text,v_screening_date,in_custody,out_custody,priors_count.1,start,end,event,two_year_recid
0,1,miguel hernandez,miguel,hernandez,2013-08-14,Male,1947-04-18,69,Greater than 45,Other,...,1,Low,2013-08-14,2014-07-07,2014-07-14,0,0,327,0,0
1,3,kevon dixon,kevon,dixon,2013-01-27,Male,1982-01-22,34,25 - 45,African-American,...,1,Low,2013-01-27,2013-01-26,2013-02-05,0,9,159,1,1
2,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,...,3,Low,2013-04-14,2013-06-16,2013-06-16,4,0,63,0,1
3,5,marcu brown,marcu,brown,2013-01-13,Male,1993-01-21,23,Less than 25,African-American,...,6,Medium,2013-01-13,NaN,NaN,1,0,1174,0,0
4,6,bouthy pierrelouis,bouthy,pierrelouis,2013-03-26,Male,1973-01-22,43,25 - 45,Other,...,1,Low,2013-03-26,NaN,NaN,2,0,1102,0,0


## 3. Inspección rápida

In [5]:
print("Columnas:", len(df.columns))
display(pd.DataFrame({
    "columna": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "nulos": [int(df[c].isna().sum()) for c in df.columns],
}).head(25))

print("\nMuestras por raza:")
if "race" in df.columns:
    display(df["race"].value_counts(dropna=False))


Columnas: 53


,columna,dtype,nulos
0,id,int64,0
1,name,object,0
2,first,object,0
3,last,object,0
4,compas_screening_date,object,0
5,sex,object,0
6,dob,object,0
7,age,int64,0
8,age_cat,object,0
9,race,object,0



Muestras por raza:


race
African-American    3696
Caucasian           2454
Hispanic             637
Other                377
Asian                 32
Native American       18
Name: count, dtype: int64

## 4. Limpieza inicial

In [6]:
def clean_compas(raw: pd.DataFrame, columnas: list) -> pd.DataFrame:
    data = raw.copy()

    # Filtrado clásico del análisis COMPAS
    # if "score_text" in data.columns:
    data = data.loc[:, columnas].copy()
    data = data.loc[
        data["days_b_screening_arrest"].between(-30, 30)
        & data["is_recid"].ne(-1)
        & data["c_charge_degree"].ne("O")
        & data["score_text"].ne("N/A")
    ].copy()

    # Mantener solo registros con seguimiento coherente
    if {"start", "end"}.issubset(data.columns):
        data = data.loc[pd.to_numeric(data["end"], errors="coerce") > pd.to_numeric(data["start"], errors="coerce")].copy()

    # Eliminar duplicados exactos
    data = data.drop_duplicates().copy()

    # Normalizar columnas de texto
    for col in data.select_dtypes(include=["object"]).columns:
        data[col] = data[col].astype(str).str.strip()

    return data

columnas = [
    "age",
    "c_charge_degree",
    "c_charge_desc",
    "race",
    "age_cat",
    "score_text",
    "sex",
    "priors_count",
    "days_b_screening_arrest",
    "decile_score",
    "is_recid",
    "two_year_recid",
    "c_jail_in",
    "c_jail_out",
]

clean_df = clean_compas(df, columnas)
print("Forma después de limpiar:", clean_df.shape)

nulls = clean_df.isna().sum().sort_values(ascending=False)
display(nulls[nulls > 0].to_frame("nulos"))


Forma después de limpiar: (6172, 14)


,nulos


## 5. Selección de variables

In [7]:
clean_df["juv_fel_count"] = df.loc[clean_df.index, "juv_fel_count"]
clean_df["juv_misd_count"] = df.loc[clean_df.index, "juv_misd_count"]
clean_df["total_number"] = clean_df["priors_count"] + clean_df["juv_fel_count"] + clean_df["juv_misd_count"]
clean_df["total_number"]

0        0
1        0
2        4
5        0
6       14
        ..
7209     0
7210     0
7211     0
7212     3
7213     2
Name: total_number, Length: 6172, dtype: int64

In [8]:
FEATURES_2 = ["age", "total_number"]

FEATURES_7 = [
    "sex",
    "age",
    "juv_fel_count",
    "juv_misd_count",
    "priors_count",
    "c_charge_degree",
    "c_charge_desc",
]

FEATURES_8 = FEATURES_7 + ["race"]
TARGET = "two_year_recid"

required_cols = sorted(set(FEATURES_8 + [TARGET]))
missing = [c for c in required_cols if c not in clean_df.columns]
if missing:
    print("Columnas faltantes:", missing)
else:
    print("Todas las columnas necesarias están presentes.")


Todas las columnas necesarias están presentes.


## 6. Filtrado para fairness (opcional)

In [13]:
def filter_race_groups(data: pd.DataFrame) -> pd.DataFrame:
    if "race" not in data.columns:
        return data.copy()
    keep = {"African-American", "Caucasian"}
    return data.loc[data["race"].isin(keep)].copy()

fair_df = filter_race_groups(clean_df)
print("Forma filtrada a Black/White:", fair_df.shape)
if "race" in fair_df.columns:
    display(fair_df["race"].value_counts())


Forma filtrada a Black/White: (5270, 53)


race
African-American    3169
Caucasian           2101
Name: count, dtype: int64

## 7. Codificación de variables categóricas

In [14]:
def encode_for_model(data: pd.DataFrame, feature_cols: list[str], target_col: str = TARGET) -> pd.DataFrame:
    frame = data.copy()

    needed = feature_cols + ([target_col] if target_col in frame.columns else [])
    frame = frame.loc[:, [c for c in needed if c in frame.columns]].copy()

    cat_cols = [c for c in feature_cols if frame[c].dtype == "object" or str(frame[c].dtype).startswith("category")]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    for col in num_cols:
        frame[col] = pd.to_numeric(frame[col], errors="coerce")
        frame[col] = frame[col].fillna(frame[col].median())

    for col in cat_cols:
        frame[col] = frame[col].fillna("Unknown").astype(str)

    if cat_cols:
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        frame[cat_cols] = enc.fit_transform(frame[cat_cols]).astype(int)

    if target_col in frame.columns:
        frame[target_col] = pd.to_numeric(frame[target_col], errors="coerce")
        frame = frame.dropna(subset=[target_col]).copy()
        frame[target_col] = frame[target_col].astype(int)

    frame = frame.dropna().reset_index(drop=True)
    return frame

df2 = encode_for_model(fair_df, FEATURES_2)
df7 = encode_for_model(fair_df, FEATURES_7)
df8 = encode_for_model(fair_df, FEATURES_8)

print("2 features:", df2.shape)
print("7 features:", df7.shape)
print("8 features:", df8.shape)
display(df8.head())


2 features: (5270, 3)
7 features: (5270, 8)
8 features: (5270, 9)


,sex,age,juv_fel_count,juv_misd_count,priors_count,c_charge_degree,c_charge_desc,race,two_year_recid
0,1,34,0,0,0,0,143,0,1
1,1,24,0,0,4,0,267,0,1
2,1,41,0,0,14,0,247,1,1
3,0,39,0,0,0,1,35,1,0
4,1,27,0,0,0,0,208,1,0


## 8. Guardado de archivos

In [15]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df2.to_csv(PROCESSED_DIR / "compas_features_2.csv", index=False)
df7.to_csv(PROCESSED_DIR / "compas_features_7.csv", index=False)
df8.to_csv(PROCESSED_FILE, index=False)

train_df, test_df = train_test_split(
    df8,
    test_size=0.2,
    random_state=42,
    stratify=df8[TARGET] if TARGET in df8.columns else None,
)

train_df.to_csv(TRAIN_FILE, index=False)
test_df.to_csv(TEST_FILE, index=False)

metadata = {
    "source_file": str(csv_path),
    "original_shape": list(df.shape),
    "clean_shape": list(clean_df.shape),
    "fair_shape": list(fair_df.shape),
    "features_2": FEATURES_2,
    "features_7": FEATURES_7,
    "features_8": FEATURES_8,
    "target": TARGET,
    "output_files": {
        "features_2": str(PROCESSED_DIR / "compas_features_2.csv"),
        "features_7": str(PROCESSED_DIR / "compas_features_7.csv"),
        "preprocessed": str(PROCESSED_FILE),
        "train": str(TRAIN_FILE),
        "test": str(TEST_FILE),
    },
}

with open(META_FILE, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Archivos generados:")
print(PROCESSED_DIR / "compas_features_2.csv")
print(PROCESSED_DIR / "compas_features_7.csv")
print(PROCESSED_FILE)
print(TRAIN_FILE)
print(TEST_FILE)
print(META_FILE)


Archivos generados:
data\processed\compas_features_2.csv
data\processed\compas_features_7.csv
data\processed\compas_preprocessed.csv
data\processed\compas_preprocessed_train.csv
data\processed\compas_preprocessed_test.csv
data\processed\compas_preprocessing_metadata.json


## 9. Verificación final

In [16]:
print("Vista previa del CSV final:")
display(pd.read_csv(PROCESSED_FILE).head())

print("\nDistribución del target en el conjunto final:")
display(pd.read_csv(PROCESSED_FILE)[TARGET].value_counts(normalize=True).rename("proporción"))


Vista previa del CSV final:


,sex,age,juv_fel_count,juv_misd_count,priors_count,c_charge_degree,c_charge_desc,race,two_year_recid
0,1,34,0,0,0,0,143,0,1
1,1,24,0,0,4,0,267,0,1
2,1,41,0,0,14,0,247,1,1
3,0,39,0,0,0,1,35,1,0
4,1,27,0,0,0,0,208,1,0



Distribución del target en el conjunto final:


two_year_recid
0    0.530361
1    0.469639
Name: proporción, dtype: float64